# NN_MODELS_COLAB

NN models (MLP, RBF network, vanilla RNN, LSTM, GRU), each as a baseline and a tuned model. Meant to run on Colab and everything else (like GLMs and other nonlin models like XGB) runs locally.

How to use (mostly just instructions for Dr. W--if any of these cause any issues, let me know and I might have a fix action. I verified this on two of my machines, so it SHOULD run cleanly for you):
1. Copy the data/processed/ folder into DRIVE_DIR on G Drive (so the drive has .../data/processed/manifest.json, data/processed/model_30SEC.csv, data/processed/model_1MIN.csv, data/processed/model_2MIN.csv)
2. Runtime --> run all. Made it so finished models are skipped, so disconnected sessions can just be rerun again.
3. The files that end up in DRIVE_DIR/results/ are what I need and will be merged with the data in my local machines results folder. STATIC_SIMULATION and NONLINEAR_MODELS pick the NN results up from there.

Same rules as local notebooks: 70/15/15 split by row, hyperparams picked on the val rows by R^2, TSCV on the first 85% as a robustness check with the chosen hyperparameters, test predictions saved separately and not used for any choice (just saved for convenience).

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import gc
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from numpy.lib.stride_tricks import sliding_window_view
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

## Configuration

In [ ]:
DRIVE_DIR = Path("/content/drive/MyDrive/Polarization Paper/notebooks/data")
DATA_DIR = DRIVE_DIR / "processed" # copy of the data/processed/
RESULTS_DIR = DRIVE_DIR / "results" # copy back into results/ when done
MODELS_DIR = DRIVE_DIR / "models" / "revised_nn"

SAMPLINGS = ["30SEC", "1MIN", "2MIN"]
TARGETS = None # None = all lead0 drift targets
MODELS_TO_RUN = ["mlp_baseline", "mlp_tuned", "rbfnn_baseline", "rbfnn_tuned", "rnn_baseline", "rnn_tuned",
                 "lstm_baseline", "lstm_tuned", "gru_baseline", "gru_tuned"]
TSCV_MODELS = ["mlp_tuned", "rbfnn_tuned", "rnn_tuned", "lstm_tuned", "gru_tuned"]
SKIP_FINISHED = True

LOOKBACK_S = 7200 # 2 h of history for the recurrent models, at every sampling rate
EPOCHS = 200
BATCH = 500
SEED = 42
CI_LEVEL = 0.95

# Leadtime runs for the locked pipeline: copy results/simulation/locked_selection.json to RESULTS_DIR/simulation/ first, or give a list of sampling, target, and model (future work--not used here)
RUN_LEAD_TIMES = True
LEAD_TIME_PIPELINES = "locked"
LEAD_TIMES_S = [120, 300, 600]

## Hyperparameters (from the original notebook)

In [ ]:
BASELINES = {
    "mlp":   {"num_layers": 2, "units": 64, "activation": "relu", "learning_rate": 1e-3},
    "rbfnn": {"K": 64, "width_scale": 1.0, "l2": 1e-4, "lr": 1e-3},
    "rnn":   {"num_layers": 1, "units": 64, "activation": "tanh", "learning_rate": 1e-3},
    "lstm":  {"num_layers": 1, "units": 64, "activation": "tanh", "learning_rate": 1e-3},
    "gru": {"num_layers": 1, "units": 64, "activation": "tanh", "learning_rate": 1e-3},
}

N_TRIALS = {"mlp": 20, "rbfnn": 15, "rnn": 20, "lstm": 20, "gru": 20}

LAYER_SPACE = {"num_layers": [1, 2, 3], "units": [16, 32, 64, 128, 256], "activation": ["tanh", "relu"], "learning_rate": [1e-4, 3e-4, 1e-3, 3e-3]}
RBF_SPACE = {"K": [16, 32, 64, 128], "width_scale": [0.5, 1.0, 1.5, 2.0], "l2": [0.0, 1e-6, 1e-5, 1e-4], "lr": [1e-4, 3e-4, 1e-3, 3e-3]}

In [ ]:
for d in (RESULTS_DIR / "nn" / "runs", RESULTS_DIR / "nn" / "tscv", RESULTS_DIR / "predictions" / "validation", RESULTS_DIR / "predictions" / "test", MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

manifest = json.loads((DATA_DIR / "manifest.json").read_text())
FEATURES = manifest["feature_columns"]
TARGET_META = {t["name"]: t for t in manifest["targets"]}
SAMPLING_S = manifest["sampling_s"]
TARGETS = TARGETS or [t["name"] for t in manifest["targets"] if t["primary"] and not t["glm_only"]]

## Data

In [ ]:
def load_dataset(sampling):
    df = pd.read_csv(DATA_DIR / f"model_{sampling}.csv")
    assert len(df) == manifest["splits"][sampling]["n"], "dataset does not match manifest"
    return df


def split_masks(df, sampling, target):
    """70/15/15 by row. Rows whose label window runs into the next split are dropped."""
    sp = manifest["splits"][sampling]
    idx = np.arange(len(df))
    end = df["current_time"].to_numpy() + TARGET_META[target]["window_end_offset_s"] * 1000
    has = df[target].notna().to_numpy()
    train = (idx < sp["n_train"]) & (end <= sp["validation_start_ms"]) & has
    val = (idx >= sp["n_train"]) & (idx < sp["n_train"] + sp["n_validation"]) & (end <= sp["test_start_ms"]) & has
    test = (idx >= sp["n_train"] + sp["n_validation"]) & has
    return train, val, test


def lookback_steps(sampling):
    return int(LOOKBACK_S / SAMPLING_S[sampling])


def metrics(y, yhat):
    return {"rmse": float(np.sqrt(mean_squared_error(y, yhat))), "r2": float(r2_score(y, yhat))}

## Scaling and model inputs

StandardScaler on X and z scoring of y, both fit on training rows only. NNs train best with inputs and outputs around zero with unit spread, according to textbook for Dr. B's class and this is what the original notebook used (the manuscripts "min-max to [-1, 1]" wording was a mistake from a previous notebook and ought to be corrected when I go back for re-writes).

Recurrent models see the lookback rows before t (just like from the original make_sequences). These rows may come from the previous split-- that is only past feature data, so val and test predictions start at the first row of their split instead of 2 h later.

In [ ]:
class Inputs:
    def __init__(self, df, target, train_mask, sequence, lookback):
        X = df[FEATURES].to_numpy(np.float32)
        y = df[target].to_numpy(np.float64)
        self.x_scaler = StandardScaler().fit(X[train_mask])
        self.X = self.x_scaler.transform(X).astype(np.float32)
        self.y_mean, self.y_std = float(y[train_mask].mean()), float(y[train_mask].std()) or 1.0
        self.y = y
        self.sequence, self.lookback = sequence, lookback
        if sequence:
            # windows[i] = X[i : i + lookback]. the window for row r is windows[r - lookback]
            self.windows = sliding_window_view(self.X, lookback, axis=0).transpose(0, 2, 1)

    def rows(self, mask):
        r = np.where(mask)[0]
        return r[r >= self.lookback] if self.sequence else r

    def x(self, rows):
        return self.windows[rows - self.lookback] if self.sequence else self.X[rows]

    def y_scaled(self, rows):
        return ((self.y[rows] - self.y_mean) / self.y_std).astype(np.float32)

    def unscale(self, yhat):
        return np.asarray(yhat, float).ravel() * self.y_std + self.y_mean

## Models (same architectures as the original notebook)

In [ ]:
def reset_tf_state(seed):
    
    keras.backend.clear_session()
    gc.collect()
    np.random.seed(seed)
    tf.random.set_seed(seed)


def build_dense(n_features, num_layers, units, activation, learning_rate):
    
    model = keras.Sequential([keras.layers.Input(shape=(n_features,))])
    
    for _ in range(int(num_layers)):
        model.add(keras.layers.Dense(int(units), activation=activation))
        
    model.add(keras.layers.Dense(1, activation="linear"))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="mse", metrics=["mae"])
    
    return model


def build_recurrent(kind, lookback, n_features, num_layers, units, activation, learning_rate, clipnorm=None):
    
    layer = {"rnn": keras.layers.SimpleRNN, "lstm": keras.layers.LSTM, "gru": keras.layers.GRU}[kind]
    model = keras.Sequential([keras.layers.Input(shape=(lookback, n_features))])
    
    for i in range(int(num_layers)):
        model.add(layer(int(units), activation=activation, return_sequences=i < int(num_layers) - 1))
        
    model.add(keras.layers.Dense(1, activation="linear"))
    opt = keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=clipnorm) if clipnorm else keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=opt, loss="mse", metrics=["mae"])
    
    return model


def kmeans_np(X, K, max_iter=50, seed=42):
    
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(X.shape[0], size=K, replace=False)].copy()
    
    for _ in range(max_iter):
        labels = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)
        new = np.vstack([X[labels == k].mean(axis=0) if np.any(labels == k) else centers[k] for k in range(K)])
        
        if np.allclose(new, centers):
            break
            
        centers = new
        
    return centers.astype(np.float32)


def compute_betas(centers, X, width_scale=1.0, eps=1e-8):
    
    d = np.sqrt(((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2).min(axis=1) + eps)
    sigma = float(np.mean(d)) * float(width_scale) or 1.0
    
    return np.full((centers.shape[0],), 1.0 / (2.0 * sigma * sigma), dtype=np.float32)

# Borrowed from the blog post referenced in the manuscript (Look this up later so I can include the citation here as well)
class RBFLayer(keras.layers.Layer):
    
    def __init__(self, centers, betas, **kwargs):
        super().__init__(**kwargs)
        self._centers_init = np.asarray(centers, dtype=np.float32)
        self._betas_init = np.asarray(betas, dtype=np.float32)

    def build(self, input_shape):
        
        self.centers = self.add_weight(name="centers", shape=self._centers_init.shape, initializer=keras.initializers.Constant(self._centers_init), trainable=False)
        self.betas = self.add_weight(name="betas", shape=self._betas_init.shape, initializer=keras.initializers.Constant(self._betas_init), trainable=False)

    def call(self, inputs):
        
        d2 = tf.reduce_sum(tf.square(tf.expand_dims(inputs, 1) - tf.expand_dims(self.centers, 0)), axis=2)
        
        return tf.exp(-self.betas * d2)


def build_rbf(n_features, centers, betas, l2, lr):
    
    inp = keras.Input(shape=(n_features,))
    out = keras.layers.Dense(1, activation="linear", kernel_regularizer=keras.regularizers.l2(l2))(RBFLayer(centers, betas)(inp))
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=float(lr)), loss="mse", metrics=["mae"])
    
    return model


def fit_model(kind, params, inp, train_rows, val_rows, seed, kmeans_seed=42, tuning=False):
    """Build/train one network and returns the fitted model."""
    reset_tf_state(seed)
    n_features = inp.X.shape[1]
    callbacks = []
    
    if kind == "rbfnn":
        centers = kmeans_np(inp.X[train_rows], int(params["K"]), seed=kmeans_seed)
        model = build_rbf(n_features, centers, compute_betas(centers, inp.X[train_rows], params["width_scale"]), params["l2"], params["lr"])
    elif kind == "mlp":
        model = build_dense(n_features, **params)
    else:
        # some of the previous notebooks I made tuned LSTM used gradient clipping and early stopping during the search (this varied greatly between notebooks tho)
        clip = 1.0 if kind == "lstm" and (tuning or params.get("_tuned")) else None
        model = build_recurrent(kind, inp.lookback, n_features, clipnorm=clip, **{k: v for k, v in params.items() if not k.startswith("_")})
        if kind == "lstm" and tuning:
            callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]
            
    model.fit(inp.x(train_rows), inp.y_scaled(train_rows), epochs=EPOCHS, batch_size=BATCH, verbose=0, validation_data=(inp.x(val_rows), inp.y_scaled(val_rows)), callbacks=callbacks)
    
    return model


def sample_params(kind, rng):
    
    space = RBF_SPACE if kind == "rbfnn" else LAYER_SPACE
    
    return {k: (int(v) if k in ("num_layers", "units", "K") else (str(v) if k == "activation" else float(v))) for k, v in ((k, rng.choice(vals)) for k, vals in space.items())}

## Train, tune, save

Tuned models: random search on the training rows, scored on the val rows; the best validation R^2 wins (raw R^2, no clipping, trials with predictions that doon't end are skipped). The winner is then
refit with seed 42, just like in the original notebooks.

In [ ]:
def run_id(sampling, target, model):
    return f"{sampling}__{target}__{model}"


PRED_COLUMNS = ["current_time", "split", "sampling", "sampling_s", "target", "metric", "window_samples", "lead_s", "variant", "window_start_offset_s", "window_end_offset_s", "model", "model_family", "y_true", "y_pred"]


def prediction_frame(df, rows, yhat, split, sampling, target, model):
    
    m = TARGET_META[target]
    out = pd.DataFrame({"current_time": df["current_time"].to_numpy()[rows], "y_true": df[target].to_numpy()[rows], "y_pred": yhat})
    out["split"], out["sampling"], out["sampling_s"], out["target"], out["model"] = split, sampling, SAMPLING_S[sampling], target, model
    
    for k in ("metric", "window_samples", "lead_s", "variant", "window_start_offset_s", "window_end_offset_s"):
        out[k] = m[k]
        
    out["model_family"] = "nn"
    
    return out[PRED_COLUMNS]


def train_one(df, sampling, target, model):
    
    kind, variant = model.split("_")
    train, val, test = split_masks(df, sampling, target)
    inp = Inputs(df, target, train, sequence=kind in ("rnn", "lstm", "gru"), lookback=lookback_steps(sampling))
    tr, va, te = inp.rows(train), inp.rows(val), inp.rows(test)
    t0 = time.time()

    if variant == "tuned":
        rng = np.random.default_rng(123)
        best = {"r2": -np.inf, "params": None}
        
        for trial in range(N_TRIALS[kind]):
            params = sample_params(kind, rng)

            # ADDED
            try:
                m = fit_model(kind, params, inp, tr, va, seed=1000 + trial, kmeans_seed=1000 + trial, tuning=True)
                yhat = inp.unscale(m.predict(inp.x(va), verbose=0))
            except Exception as e:
                print(f"  trial {trial + 1} skipped: {type(e).__name__}")
                continue
                
            if not np.isfinite(yhat).all():
                continue
                
            r2 = r2_score(inp.y[va], yhat)
            
            if r2 > best["r2"]:
                best = {"r2": r2, "params": params}
                
        params = best["params"]
        
        if params is None:
            raise RuntimeError(f"no successful tuning trial for {model}")
            
        if kind == "lstm":
            params = {**params, "_tuned": True}
            
    else:
        params = dict(BASELINES[kind])

    #final = fit_model(kind, params, inp, tr, va, seed=SEED)
    #yhat = {s: inp.unscale(final.predict(inp.x(r), verbose=0)) for s, r in (("train", tr), ("validation", va), ("test", te))}
    # fixed to handle crashing--likely from exploding gradients in recurrent layers that ended up using relu act funcs
    for refit_seed in (SEED, SEED + 1, SEED + 2):
        final = fit_model(kind, params, inp, tr, va, seed=refit_seed)
        yhat = {s: inp.unscale(final.predict(inp.x(r), verbose=0)) for s, r in (("train", tr), ("validation", va), ("test", te))}
        if all(np.isfinite(v).all() for v in yhat.values()):
            break
    else:
        raise RuntimeError(f"final refit diverged for seeds {SEED}-{SEED + 2}")
    m_tr, m_va, m_te = metrics(inp.y[tr], yhat["train"]), metrics(inp.y[va], yhat["validation"]), metrics(inp.y[te], yhat["test"])

    rid = run_id(sampling, target, model)
    final.save_weights(str(MODELS_DIR / f"{rid}.weights.h5"))
    (MODELS_DIR / f"{rid}.params.json").write_text(json.dumps(params))
    
    pd.concat([prediction_frame(df, tr, yhat["train"], "train", sampling, target, model), prediction_frame(df, va, yhat["validation"], "validation", sampling, target, model)]).to_csv(RESULTS_DIR / "predictions" / "validation" / f"{rid}.csv", index=False)
    prediction_frame(df, te, yhat["test"], "test", sampling, target, model).to_csv(RESULTS_DIR / "predictions" / "test" / f"{rid}.csv", index=False)
    
    #record = {"sampling frequency": sampling, "model_name": model, "target_name": target, "params": json.dumps(params), "n_train": len(tr), "n_validation": len(va), "n_test": len(te), "training_rmse": m_tr["rmse"], "training_r2": m_tr["r2"], "validation_rmse": m_va["rmse"], "validation_r2": m_va["r2"], "test_rmse": m_te["rmse"], "test_r2": m_te["r2"], "seconds": round(time.time() - t0, 1)}
    record = {"sampling frequency": sampling, "model_name": model, "target_name": target, "params": json.dumps(params), "n_train": len(tr), "n_validation": len(va), "n_test": len(te), "training_rmse": m_tr["rmse"], "training_r2": m_tr["r2"], "validation_rmse": m_va["rmse"], "validation_r2": m_va["r2"], "test_rmse": m_te["rmse"], "test_r2": m_te["r2"], "seconds": round(time.time() - t0, 1), "refit_seed": refit_seed}
    (RESULTS_DIR / "nn" / "runs" / f"{rid}.json").write_text(json.dumps(record, indent=2))
    
    return record

## TSCV robustness check

Same folds as the notebooks on my local machine (24 h / 12 h forward chaining on the first 85%), the chosen hyperparameters, and the same 2 h lookback (fixed for consistency this time, lol). Scalers are refit on each folds training rows.

In [ ]:
def tscv_folds(df, sampling, target):
    
    sp, cfg = manifest["splits"][sampling], manifest["tscv"]
    n_rows = sp["n_train"] + sp["n_validation"]
    rows_per_h = 3600 / SAMPLING_S[sampling]
    first, val_len, step = (int(cfg[k] * rows_per_h) for k in ("initial_train_h", "val_h", "step_h"))
    min_last = int(cfg["min_last_val_h"] * rows_per_h)
    t = df["current_time"].to_numpy()
    end = t + TARGET_META[target]["window_end_offset_s"] * 1000
    has, idx = df[target].notna().to_numpy(), np.arange(len(df))
    folds, train_end = [], first
    
    while train_end < n_rows:
        val_end = min(train_end + val_len, n_rows)
        
        if val_end - train_end < min_last:
            break
            
        folds.append(((idx < train_end) & (end <= t[train_end]) & has, (idx >= train_end) & (idx < val_end) & (end <= sp["test_start_ms"]) & has))
        train_end += step
        
    return folds


def run_tscv(df, sampling, target, model, params):
    
    kind = model.split("_")[0]
    t = df["current_time"].to_numpy()
    iso = lambda ms: pd.Timestamp(int(ms), unit="ms", tz="UTC").isoformat()
    rows = []
    
    for k, (train, val) in enumerate(tscv_folds(df, sampling, target), 1):
        inp = Inputs(df, target, train, sequence=kind in ("rnn", "lstm", "gru"), lookback=lookback_steps(sampling))
        tr, va = inp.rows(train), inp.rows(val)
        #m = fit_model(kind, params, inp, tr, va, seed=SEED + k)
        #yhat = inp.unscale(m.predict(inp.x(va), verbose=0))
        for fold_seed in (SEED + k, SEED + k + 10, SEED + k + 20):
            m = fit_model(kind, params, inp, tr, va, seed=fold_seed)
            yhat = inp.unscale(m.predict(inp.x(va), verbose=0))
            if np.isfinite(yhat).all():
                break
        else:
            raise RuntimeError(f"TSCV fold {k} diverged for seeds {SEED + k}, {SEED + k + 10}, {SEED + k + 20}")
        #rows.append({"sampling frequency": sampling, "model_name": model, "target_name": target, "fold": k, "train_start": iso(t[tr].min()), "train_end": iso(t[tr].max()), "val_start": iso(t[va].min()), "val_end": iso(t[va].max()), "n_train": len(tr), "n_val": len(va), **metrics(inp.y[va], yhat), "y_true": json.dumps(inp.y[va].tolist()), "y_pred": json.dumps(yhat.tolist())})
        rows.append({"sampling frequency": sampling, "model_name": model, "target_name": target, "fold": k, "seed": fold_seed, "train_start": iso(t[tr].min()), "train_end": iso(t[tr].max()), "val_start": iso(t[va].min()), "val_end": iso(t[va].max()), "n_train": len(tr), "n_val": len(va), **metrics(inp.y[va], yhat), "y_true": json.dumps(inp.y[va].tolist()), "y_pred": json.dumps(yhat.tolist())})
    
    return pd.DataFrame(rows)

## Run

In [ ]:
def run(configs):
    datasets = {}
    
    for sampling, target, model in configs:
        rid = run_id(sampling, target, model)
        run_file, tscv_file = RESULTS_DIR / "nn" / "runs" / f"{rid}.json", RESULTS_DIR / "nn" / "tscv" / f"{rid}.csv"
        
        if sampling not in datasets:
            datasets[sampling] = load_dataset(sampling)
        df = datasets[sampling]
        
        #if not (SKIP_FINISHED and run_file.exists()):
        #    rec = train_one(df, sampling, target, model)
        #    print(f"{rid:60s} val R^2={rec['validation_r2']:.4f}  RMSE={rec['validation_rmse']:.5f}  ({rec['seconds']} s)")
        #     
        #if model in TSCV_MODELS and not (SKIP_FINISHED and tscv_file.exists()):
        #    params = json.loads(json.loads(run_file.read_text())["params"])
        #    run_tscv(df, sampling, target, model, params).to_csv(tscv_file, index=False)
        try:
            step = "train"
            if not (SKIP_FINISHED and run_file.exists()):
                # Dr. W: You can put your ensure_drive_connected() func here (I only saw part of it from your screenshot):
                rec = train_one(df, sampling, target, model)
                print(f"{rid:60s} val R^2={rec['validation_r2']:.4f}  RMSE={rec['validation_rmse']:.5f}  ({rec['seconds']} s)")

            step = "tscv"
            if model in TSCV_MODELS and not (SKIP_FINISHED and tscv_file.exists()):
                params = json.loads(json.loads(run_file.read_text())["params"])
                run_tscv(df, sampling, target, model, params).to_csv(tscv_file, index=False)
        except Exception as e:
            failure_file = RESULTS_DIR / "nn" / "failures.csv"
            pd.DataFrame([{"time_utc": pd.Timestamp.now(tz="UTC").isoformat(), "run_id": rid, "step": step,
                           "error": type(e).__name__, "message": str(e)[:300]}]).to_csv(
                failure_file, mode="a", header=not failure_file.exists(), index=False)
            print(f"{rid:60s} FAILED at {step}: {type(e).__name__}: {e}")


run([(s, t, m) for s in SAMPLINGS for t in TARGETS for m in MODELS_TO_RUN])

## Lead-time runs (L = 2 / 5 / 10 min) (NOT RUN HERE)

In [ ]:
def lead_target(target, lead_s):
    base = TARGET_META[target]["base"]
    return next((t for t, m in TARGET_META.items() if m["base"] == base and m["lead_s"] == lead_s), None)


lock_file = RESULTS_DIR / "simulation" / "locked_selection.json"
if RUN_LEAD_TIMES:
    pipelines = ([json.loads(lock_file.read_text())] if lock_file.exists() else []) if LEAD_TIME_PIPELINES == "locked" else LEAD_TIME_PIPELINES
    configs = [(p["sampling"], lead_target(p["target"], L), p["model"]) for p in pipelines if p["model"] in MODELS_TO_RUN for L in LEAD_TIMES_S if lead_target(p["target"], L)]
    if not configs:
        print("No NN pipeline to run lead times for (no lock file, or the locked model is not an NN).")
    run(configs)

## Collect results

validation_results.csv is what the notebooks on my machine read. test_results.csv is just for reporting only.

In [ ]:
def summarize_tscv(fold_df):
    out = []
    
    for key, g in fold_df.groupby(["sampling frequency", "model_name", "target_name"]):
        row = dict(zip(["sampling frequency", "model_name", "target_name"], key), n_folds=len(g))
        
        for m in ("rmse", "r2"):
            v = g[m].to_numpy(float)
            sd = v.std(ddof=1) if len(v) > 1 else np.nan
            half = stats.t.ppf(0.5 + CI_LEVEL / 2, len(v) - 1) * sd / math.sqrt(len(v)) if len(v) > 1 else np.nan
            row.update({f"mean_{m}": v.mean(), f"median_{m}": np.median(v), f"std_{m}": sd, f"{m}_ci_low": v.mean() - half, f"{m}_ci_high": v.mean() + half})
            
        y_all = np.concatenate([json.loads(s) for s in g["y_true"]])
        p_all = np.concatenate([json.loads(s) for s in g["y_pred"]])
        row.update({f"pooled_{k}": v for k, v in metrics(y_all, p_all).items()})
        out.append(row)
        
    return pd.DataFrame(out)


runs = pd.DataFrame([json.loads(p.read_text()) for p in sorted((RESULTS_DIR / "nn" / "runs").glob("*.json"))])
test_cols = ["test_rmse", "test_r2", "n_test"]
runs.drop(columns=test_cols).to_csv(RESULTS_DIR / "nn" / "validation_results.csv", index=False)
runs[["sampling frequency", "model_name", "target_name"] + test_cols].to_csv(RESULTS_DIR / "nn" / "test_results.csv", index=False)

tscv_files = sorted((RESULTS_DIR / "nn" / "tscv").glob("*.csv"))
if tscv_files:
    folds = pd.concat([pd.read_csv(p) for p in tscv_files], ignore_index=True)
    folds.drop(columns=["y_true", "y_pred"]).to_csv(RESULTS_DIR / "nn" / "tscv_folds.csv", index=False)
    summarize_tscv(folds).to_csv(RESULTS_DIR / "nn" / "tscv_summary.csv", index=False)

runs.drop(columns=test_cols).sort_values("validation_r2", ascending=False).head(10)